<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aibizx/learn-ml-algorithms-2017-notebooks/blob/main/04-logistic-regression.ipynb)

_Part of the [AI/Biz books](https://www.ai.biz/books/learn-ml-algorithms-2017/) collection._

# Chapter 4 — Logistic Regression and Classification

Companion to [the chapter](https://www.ai.biz/books/learn-ml-algorithms-2017/logistic-regression-and-classification/).

A new world where the label is yes or no: will a subscriber renew? We write the true probability ourselves, so every model can be marked against it.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def make_subscribers(n, seed=0):
    """Hours of use per week, months as a customer, and whether they renewed."""
    rng = np.random.default_rng(seed)
    hours = rng.uniform(0, 10, n)
    months = rng.uniform(1, 36, n)
    logit = -3.0 + 0.6 * hours + 0.05 * months        # the true log-odds of renewing
    p = sigmoid(logit)
    renewed = (rng.uniform(size=n) < p).astype(int)   # the coin is flipped with that probability
    return np.column_stack([hours, months]), renewed, p

X, y, p_true = make_subscribers(400, seed=1)
X_test, y_test, p_test = make_subscribers(4000, seed=2)
print(f'{len(y)} subscribers, {y.mean():.0%} renewed')
for i in range(3):
    print(f'  {X[i, 0]:.1f} h/week, {X[i, 1]:4.1f} months -> true chance {p_true[i]:.2f}, renewed {y[i]}')

## 1. Why a straight line will not do

In [ ]:
from sklearn.linear_model import LinearRegression

line = LinearRegression().fit(X, y)
pred = line.predict(X_test)
print(f'straight-line "probabilities" range from {pred.min():.2f} to {pred.max():.2f}')
print(f'{(pred < 0).mean():.1%} of subscribers get a negative chance of renewing')
print(f'{(pred > 1).mean():.1%} get a chance above one')

## 2. The sigmoid turns any number into a probability

In [ ]:
for z in [-6, -3, -1, 0, 1, 3, 6]:
    print(f'z = {z:>3}  sigmoid {sigmoid(z):.3f}   odds {np.exp(z):8.3f} to 1')

## 3. Log loss, and why accuracy cannot be the loss

In [ ]:
def log_loss(p, y):
    eps = 1e-12
    return -np.mean(y * np.log(p + eps) + (1 - y) * np.log(1 - p + eps))

print('what one confident wrong answer costs:')
for p in [0.5, 0.9, 0.99, 0.999]:
    print(f'  predicted {p:.3f} for a subscriber who left: log loss {-np.log(1 - p):6.2f}')

In [ ]:
def add_intercept(X):
    return np.column_stack([np.ones(len(X)), X])

Xb = add_intercept(X)

def loss(w):
    return log_loss(sigmoid(Xb @ w), y)

def gradient(w):
    return Xb.T @ (sigmoid(Xb @ w) - y) / len(y)

def accuracy_loss(w):
    return 1 - np.mean((sigmoid(Xb @ w) > 0.5) == y)

w0 = np.array([0.0, 0.1, 0.0])
for name, f in [('log loss', loss), ('error rate', accuracy_loss)]:
    nudged = [f(w0 + np.array([0, h, 0])) - f(w0) for h in [1e-4, 1e-3, 1e-2]]
    print(f'{name:<11} change when the hours weight moves by 0.0001, 0.001, 0.01:', ['%.2e' % d for d in nudged])

## 4. Gradient descent, again

In [ ]:
def numerical_gradient(f, w, h=1e-6):
    g = np.zeros_like(w)
    for i in range(len(w)):
        step = np.zeros_like(w); step[i] = h
        g[i] = (f(w + step) - f(w - step)) / (2 * h)
    return g

print('formula  :', np.round(gradient(w0), 5))
print('numerical:', np.round(numerical_gradient(loss, w0), 5))

In [ ]:
mean, std = X.mean(axis=0), X.std(axis=0)
Xs = add_intercept((X - mean) / std)

def loss_s(w): return log_loss(sigmoid(Xs @ w), y)
def grad_s(w): return Xs.T @ (sigmoid(Xs @ w) - y) / len(y)

w = np.zeros(3)
for t in range(1, 3001):
    w = w - 0.5 * grad_s(w)
    if t in (1, 10, 100, 1000, 3000):
        print(f'step {t:>4}  log loss {loss_s(w):.4f}')

# Undo the standardisation so the weights are per hour and per month again
w_hours, w_months = w[1] / std[0], w[2] / std[1]
w_int = w[0] - w_hours * mean[0] - w_months * mean[1]
print(f'\nlearned : intercept {w_int:.2f}, hours {w_hours:.3f}, months {w_months:.3f}')
print('truth   : intercept -3.00, hours 0.600, months 0.050')

## 5. Reading the weights as odds

In [ ]:
print(f'one more hour a week multiplies the odds of renewing by {np.exp(w_hours):.2f}')
print(f'one more year as a customer multiplies them by {np.exp(12 * w_months):.2f}')

## 6. The one line you would ship

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

clf = make_pipeline(StandardScaler(), LogisticRegression(C=1e6)).fit(X, y)
lr = clf[-1]
print('scikit-learn weights (standardised):', np.round(lr.coef_[0], 3), ' ours:', np.round(w[1:], 3))
p_hat = clf.predict_proba(X_test)[:, 1]
print(f'log loss on new subscribers: ours {log_loss(sigmoid(add_intercept((X_test - mean) / std) @ w), y_test):.4f}, '
      f'scikit-learn {log_loss(p_hat, y_test):.4f}, the true probabilities {log_loss(p_test, y_test):.4f}')

## 7. The threshold is a business decision

In [ ]:
def confusion(p, y, threshold):
    pred = (p >= threshold).astype(int)
    tp = np.sum((pred == 1) & (y == 1)); fp = np.sum((pred == 1) & (y == 0))
    fn = np.sum((pred == 0) & (y == 1)); tn = np.sum((pred == 0) & (y == 0))
    return tp, fp, fn, tn

# Flip the question: we want to find the subscribers who will LEAVE, and offer them a discount.
p_leave = 1 - p_hat
y_leave = 1 - y_test
cost_offer, lost_revenue = 20, 120     # a discount costs £20; a customer who leaves unwarned costs £120

print(f'{"threshold":>9} {"flagged":>8} {"caught":>7} {"missed":>7} {"precision":>9} {"recall":>7} {"cost":>9}')
for t in [0.2, 0.3, 0.4, 0.5, 0.6, 0.8]:
    tp, fp, fn, tn = confusion(p_leave, y_leave, t)
    cost = cost_offer * (tp + fp) + lost_revenue * fn
    print(f'{t:>9.1f} {tp + fp:>8} {tp:>7} {fn:>7} {tp / max(tp + fp, 1):>9.2f} {tp / (tp + fn):>7.2f} {cost:>9,}')
print(f'\noffer everyone a discount: cost {cost_offer * len(y_leave):,}   offer nobody: cost {lost_revenue * y_leave.sum():,}')

In [ ]:
# The threshold that the costs imply: act when the expected loss from waiting exceeds the cost of acting
print(f'break-even probability of leaving: {cost_offer / lost_revenue:.3f}')

## 8. Is a 0.7 really a 70% chance?

In [ ]:
bins = np.linspace(0, 1, 11)
which = np.digitize(p_hat, bins) - 1
print(f'{"predicted":>12} {"actual rate":>12} {"count":>6}')
for b in range(10):
    m = which == b
    if m.sum() > 20:
        print(f'{bins[b]:>5.1f}-{bins[b + 1]:<5.1f} {y_test[m].mean():>12.2f} {m.sum():>6}')

## 9. Imbalance: when leavers are rare

In [ ]:
rng = np.random.default_rng(7)
keep = (y == 1) | (rng.uniform(size=len(y)) < 0.15)      # keep every renewer and 15% of leavers
X_rare, y_rare = X[keep], y[keep]
print(f'{len(y_rare)} subscribers, {1 - y_rare.mean():.1%} leave')

plain = make_pipeline(StandardScaler(), LogisticRegression()).fit(X_rare, y_rare)
weighted = make_pipeline(StandardScaler(), LogisticRegression(class_weight='balanced')).fit(X_rare, y_rare)
keep_t = (y_test == 1) | (rng.uniform(size=len(y_test)) < 0.15)
Xr_t, yr_t = X_test[keep_t], y_test[keep_t]

for name, m in [('plain', plain), ('class_weight=balanced', weighted)]:
    pred = m.predict(Xr_t)
    leavers_found = np.sum((pred == 0) & (yr_t == 0)) / np.sum(yr_t == 0)
    print(f'{name:<22} accuracy {np.mean(pred == yr_t):.3f}   leavers found {leavers_found:.2f}')
print(f'{"always say renew":<22} accuracy {yr_t.mean():.3f}   leavers found 0.00')

## 10. More than two classes

In [ ]:
from sklearn.datasets import load_iris

iris = load_iris()
multi = make_pipeline(StandardScaler(), LogisticRegression()).fit(iris.data, iris.target)
probs = multi.predict_proba(iris.data[[0, 60, 120]])
print('three flowers, probability of each species:')
print(np.round(probs, 3), '\nrows sum to', np.round(probs.sum(axis=1), 3))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
h = np.linspace(0, 10, 200); m = np.linspace(1, 36, 200)
H, M = np.meshgrid(h, m)
grid = np.column_stack([H.ravel(), M.ravel()])
P = clf.predict_proba(grid)[:, 1].reshape(H.shape)
cs = ax.contourf(H, M, P, levels=np.linspace(0, 1, 11), cmap='RdYlGn', alpha=0.8)
ax.scatter(X[:, 0], X[:, 1], c=y, cmap='RdYlGn', edgecolor='k', s=18)
ax.contour(H, M, P, levels=[0.5], colors='black')
plt.colorbar(cs, label='probability of renewing')
ax.set_xlabel('hours per week'); ax.set_ylabel('months as a customer')
plt.show()

## Try it yourself

1. Change the costs in section 7 so a lost customer is worth £40 rather than £120. Where does the best threshold move, and does the break-even formula agree?
2. Train on 40 subscribers instead of 400. How far do the weights drift from the truth, and does the calibration table in section 8 still hold?
3. In section 9, try `class_weight={0: 10, 1: 1}` and `{0: 3, 1: 1}`. Which finds more leavers, and what does it cost in accuracy?